In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./FAQ.pdf')
pages = loader.load()

full_text = "\n".join([page.page_content for page in pages])

print(full_text[:500])

c:\Users\Admin\miniconda3\envs\pystudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FAQ 질문과 답변Q. 이 챗봇은 어떤 일을 해 주나요?A. 이 챗봇은 어르신의 복지서비스·복약 정보·상담 기록을 바탕으로 보호자의 질문에 답하고, 필요한 경우 담당 상담사에게 연결을 도와주는 정보 안내·업무 지원용 도구다.​Q. 의료 진단이나 약 복용량을 대신 알려주나요?A. 챗봇은 질병 진단이나 구체적인 약 복용량을 안내하지 않고, 일반적인 정보와 공공 가이드라인 수준의 설명만 제공하며, 위험하거나 애매한 상황에서는 항상 담당 의료진이나 상담사에게 문의하도록 안내한다.​Q. 어르신 개인 정보와 상담 내용은 어떻게 보호되나요?A. 어르신·보호자 정보는 최소한의 필드만 저장하고, 접근 권한이 있는 계정(담당 상담사, 해당 보호자)만 Q&A와 상담 기록을 조회할 수 있도록 설계하며, 민감한 내용은 접근 로그와 함께 감사 기록에 남긴다.​​Q. 과거 상담 내용을 바탕으로 답해 준다고 했는데, 그 기록은 얼마나 오래 보관되나요?A. 개별 대화 전체를 계속 쌓아 두기보다는 일정 기간이 지나


In [2]:
import re 

pattern = r'(?=Q\d+\.)'

chunks = [chunk.strip() for chunk in re.split(pattern, full_text) if chunk.strip()]

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk[:100] + "...") 
    print("\n")

--- Chunk 1 ---
FAQ 질문과 답변Q. 이 챗봇은 어떤 일을 해 주나요?A. 이 챗봇은 어르신의 복지서비스·복약 정보·상담 기록을 바탕으로 보호자의 질문에 답하고, 필요한 경우 담당 상담사에게 연...




In [3]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('./qna.csv', encoding='utf-8')

documents = loader.load()

print(len(documents))
print(documents[0])

4
page_content='Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 아버지께서 드시는 약을 챗봇에 등록해 두고 싶은데
집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 어떤 정보를 적어야 나중에 도움이 될까요?A: 약 이름(상품명)
차: 성분명(가능하다면)
탄산음료)을 줄이고: 1회 복용량
잠들기 1시간 전에는 TV·스마트폰 사용을 줄여 조용하고 어둡게 환경을 정리해 주세요. 필요하면 일정 시간에 가벼운 스트레칭이나 따뜻한 물 족욕을 함께 해 드리면 수면 리듬을 맞추는 데 도움이 될 수 있습니다.: 하루 몇 번/언제 복용하는지
None: 처방한 병원·의사 이름,주요 부작용 안내 문구 정도를 기록해 두면 좋습니다. 나중에 복용 시간 알림이나 “이 약은 언제까지 먹어야 하나요?” 같은 질문에 답할 때 이 정보가 활용됩니다.' metadata={'source': './qna.csv', 'row': 0}


In [4]:
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [28]:
%pip install langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_core.documents import Document

# FAQ chunks to documents
faq_documents = [Document(page_content=chunk, metadata={"source": "FAQ.pdf", "type": "faq"}) for chunk in chunks]

# Combine all documents
all_documents = documents + faq_documents

print(f"Q&A Documents: {len(documents)}")
print(f"FAQ Documents: {len(faq_documents)}")
print(f"Total Documents: {len(all_documents)}")

Q&A Documents: 4
FAQ Documents: 1
Total Documents: 5


In [8]:
from pinecone import Pinecone, ServerlessSpec
import os

# 1. Pinecone 클라이언트 초기화
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "skt-index"  # 코드에서 사용하신 이름과 동일하게 설정

# 2. 인덱스가 있는지 확인하고, 없으면 생성
if index_name not in pc.list_indexes().names():
    print(f"Index '{index_name}'가 없어서 새로 생성합니다...")
    pc.create_index(
        name=index_name,
        dimension=1536, # OpenAI text-embedding-3-small 모델의 차원 수
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1" # Pinecone 무료 버전(Starter) 기본 리전
        ) 
    )
    print(f"Index '{index_name}' 생성 완료!")
else:
    print(f"Index '{index_name}'가 이미 존재합니다.")

Index 'skt-index'가 없어서 새로 생성합니다...
Index 'skt-index' 생성 완료!


In [9]:
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv
import os

# Force reload .env file to ensure latest keys are loaded
load_dotenv(override=True)

# Load API Key explicitly
pinecone_api_key = os.getenv('PINECONE_API_KEY')
INECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME')
PINECONE_NAMESPACE = os.getenv('PINECONE_NAMESPACE')
print(f"DEBUG: PINECONE_API_KEY loaded? {bool(pinecone_api_key)}")

if not pinecone_api_key:
    print("Warning: PINECONE_API_KEY not found in environment variables. Please check .env file.")

# Index name
index_name = "skt-index"

# Upload to Pinecone
if pinecone_api_key: 
    vectorstore = PineconeVectorStore.from_documents(
        documents=all_documents,
        embedding=embeddings,
        index_name=index_name,
        pinecone_api_key=pinecone_api_key
    )
    print(f"Successfully uploaded {len(all_documents)} documents to Pinecone index '{index_name}'")


DEBUG: PINECONE_API_KEY loaded? True
Successfully uploaded 5 documents to Pinecone index 'skt-index'


In [10]:
import numpy as np
from langchain_openai.embeddings import OpenAIEmbeddings

# 이미 위에서 만든 embeddings 재사용
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def embed_text(text: str) -> np.ndarray:
    return np.array(embeddings.embed_query(text), dtype="float32")

# 1) FAQ 벡터화 (chunks 리스트 사용)
faq_docs = []
for i, chunk in enumerate(chunks):
    faq_docs.append({
        "id": f"FAQ_{i+1}",
        "type": "FAQ",
        "text": chunk,
        "vector": embed_text(chunk)
    })

print(f"FAQ docs: {len(faq_docs)}개")


FAQ docs: 1개


In [11]:
# 2) Q&A 벡터화 (qna.csv에서 로드한 documents 사용)
# documents[n].page_content 안에 Q/A 텍스트가 들어 있다고 가정
qna_docs = []
for i, doc in enumerate(documents):
    qna_docs.append({
        "id": f"QNA_{i+1}",
        "type": "QNA",
        "text": doc.page_content,
        "vector": embed_text(doc.page_content)
    })

print(f"Q&A docs: {len(qna_docs)}개")


Q&A docs: 4개


In [12]:
all_docs = faq_docs + qna_docs
print(f"총 문서 수: {len(all_docs)}개")


총 문서 수: 5개


In [13]:
from typing import List, Dict, Tuple

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    if a.ndim == 1:
        a = a.reshape(1, -1)
    if b.ndim == 1:
        b = b.reshape(1, -1)
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return float(np.dot(a_norm, b_norm.T)[0][0])

def search_top_k(query: str, k: int = 3) -> List[Tuple[float, Dict]]:
    q_vec = embed_text(query)
    scored: List[Tuple[float, Dict]] = []

    for doc in all_docs:
        sim = cosine_similarity(q_vec, doc["vector"])
        scored.append((sim, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]


In [16]:
test_queries = [
    "상담사는 일주일에 몇 번 방문해?",
    "엄마 어제 병원에 다녀오셨어?",
    "아빠 식사는 하셨어?"
]

for q in test_queries:
    print(f"\n=== Query: {q} ===")
    results = search_top_k(q, k=3)
    for rank, (score, doc) in enumerate(results, start=1):
        print(f"[{rank}] type={doc['type']}, id={doc['id']}, score={score:.4f}")
        print(doc["text"][:120].replace("\n", " ") + "...")
        print()



=== Query: 상담사는 일주일에 몇 번 방문해? ===
[1] type=QNA, id=QNA_2, score=0.3073
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 주간보호센터 이용을 고민 중인데 집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 어머니 치매 초기에 이용하면 어떤 장단점이 있을까...

[2] type=QNA, id=QNA_3, score=0.2658
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 지난번 상담에서 어머니 영양제를 잠시 중단하자는 말씀을 들었는데 집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 다시 시작해도 되...

[3] type=QNA, id=QNA_4, score=0.2559
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 제가 직장 때문에 낮에 집을 자주 비우는데 집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 그 시간에 어머니 안전을 위해 챗봇과 ...


=== Query: 엄마 어제 병원에 다녀오셨어? ===
[1] type=QNA, id=QNA_3, score=0.3883
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 지난번 상담에서 어머니 영양제를 잠시 중단하자는 말씀을 들었는데 집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 다시 시작해도 되...

[2] type=QNA, id=QNA_4, score=0.3818
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 제가 직장 때문에 낮에 집을 자주 비우는데 집에서 도와드릴 수 있는 방법이 있을까요?A: 우선 저녁 카페인(커피: 그 시간에 어머니 안전을 위해 챗봇과 ...

[3] type=QNA, id=QNA_1, score=0.3623
Q: 우리 어머니가 최근에 밤에 자주 깨고 뒤척이시는데: Q: 아버지께서 드시는 약을 챗봇에 등록해 두고 싶은데 집에서 도와드릴 수 있는 방법이 있을까요